In [1]:
import numpy as np
import pandas as pd
from sklearn.utils import resample
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import string as s
import re as r
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,classification_report
import joblib as j
from sklearn.metrics import  recall_score, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
import json
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Language Detection.csv')

In [4]:
df.shape

(10337, 2)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10337 entries, 0 to 10336
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Text      10337 non-null  object
 1   Language  10337 non-null  object
dtypes: object(2)
memory usage: 161.6+ KB


In [6]:
df.isnull().sum()

,0
Text,0
Language,0


In [7]:
unique_counts = df.nunique()
print(unique_counts)

Text        10267
Language       17
dtype: int64


In [8]:
print(df['Language'].unique())

['English' 'Malayalam' 'Hindi' 'Tamil' 'Portugeese' 'French' 'Dutch'
 'Spanish' 'Greek' 'Russian' 'Danish' 'Italian' 'Turkish' 'Sweedish'
 'Arabic' 'German' 'Kannada']


In [9]:
df['Language'].value_counts()

,count
Language,
English,1385
French,1014
Spanish,819
Portugeese,739
Italian,698
Russian,692
Sweedish,676
Malayalam,594
Dutch,546


In [10]:
df_upsampled = pd.DataFrame()

for language in df['Language'].unique():
    language_df = df[df['Language'] == language]

    if not language_df.empty:
        count = language_df['Language'].value_counts().values[0]

        if count < 1385:
            upsampled_df = resample(language_df,
                                    replace=True,
                                    n_samples=1385,
                                    random_state=42)
            df_upsampled = pd.concat([df_upsampled, upsampled_df])
        else:
            df_upsampled = pd.concat([df_upsampled, language_df])

In [11]:
df_upsampled.reset_index(drop=True, inplace=True)

df_upsampled['Language'].value_counts()

,count
Language,
English,1385
Malayalam,1385
Hindi,1385
Tamil,1385
Portugeese,1385
French,1385
Dutch,1385
Spanish,1385
Greek,1385


In [12]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    """Map POS tag to WordNet format for accurate lemmatization"""
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def preprocessing(x):
    pun = s.punctuation
    tokens = word_tokenize(x)
    tokens = [w for w in tokens if w not in pun and w.strip()]
    pos_tags = nltk.pos_tag(tokens)
    lemmatized = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in pos_tags
    ]
    return " ".join(lemmatized)

In [13]:
df_upsampled['Text'] = df_upsampled['Text'].apply(preprocessing)

In [14]:
df_upsampled

,Text,Language
0,Nature in the broad sense be the natural physi...,English
1,`` Nature '' can refer to the phenomenon of th...,English
2,The study of nature be a large if not the only...,English
3,Although human be part of nature human activit...,English
4,1 The word nature be borrow from the Old Frenc...,English
...,...,...
23540,ಪರೀಕ್ಷೆಯಲ್ಲಿ ಒಂದೇ ಪ್ರಶ್ನೆಗೆ ಉತ್ತರಿಸಲು ಅಥವಾ ನಿಷ...,Kannada
23541,ಇದರ ಅಡಿಯಲ್ಲಿ ನನಗೆ ಈ 10 ಪದಗಳಲ್ಲಿ ಯಾವುದು ನಿಮ್ಮ ನ...,Kannada
23542,ನಿಮ್ಮನ್ನು ತೊಂದರೆಗೊಳಿಸಿದ್ದಕ್ಕೆ ಕ್ಷಮಿಸಿ,Kannada
23543,ನಾನು ಅದರ ಬಗ್ಗೆ ನಿಜವಾಗಿಯೂ ವಿಷಾದಿಸುತ್ತೇನೆ,Kannada


In [15]:
x=df_upsampled['Text']
y=df_upsampled['Language']

In [16]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [17]:
vector=TfidfVectorizer()
x_train_tidf=vector.fit_transform(x_train)
x_test_tidf=vector.transform(x_test)

In [18]:
lr=LogisticRegression()
lr.fit(x_train_tidf,y_train)

LogisticRegression()

In [19]:
y_pred=lr.predict(x_test_tidf)
y_tpre=lr.predict(x_train_tidf)

In [20]:
y_train_pred = lr.predict(x_train_tidf)
y_test_pred = lr.predict(x_test_tidf)
print("Training Set Metrics:")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("Precision:", precision_score(y_train, y_train_pred, average='weighted'))
print("Recall:", recall_score(y_train, y_train_pred, average='weighted'))
print("F1-score:", f1_score(y_train, y_train_pred, average='weighted'))
print("-" * 30)

Training Set Metrics:
Accuracy: 0.9957528137608834
Precision: 0.9959339898843492
Recall: 0.9957528137608834
F1-score: 0.9957816397297763
------------------------------


In [21]:
print("Test Set Metrics:")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_test_pred, average='weighted'))
print("F1-score:", f1_score(y_test, y_test_pred, average='weighted'))

Test Set Metrics:
Accuracy: 0.9887449564663411
Precision: 0.9900320987446379
Recall: 0.9887449564663411
F1-score: 0.9890392376937965


In [22]:
sentence4 = "क्या आप अंग्रेज़ी बोलते हैं"
preprocessed_sentence4 = preprocessing(sentence4)
X_sentence4 = vector.transform([preprocessed_sentence4])
prediction4 = lr.predict(X_sentence4)
print(f"Sentence: '{sentence4}' -> Predicted Language: {prediction4[0]}")

Sentence: 'क्या आप अंग्रेज़ी बोलते हैं' -> Predicted Language: Hindi


In [23]:
!pip install transformers sentencepiece -q

In [34]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline, NllbTokenizerFast

print("⏳ Loading NLLB-200 translation model...")

model_name = "facebook/nllb-200-1.3B"
tokenizer = NllbTokenizerFast.from_pretrained(model_name) # Use NllbTokenizerFast directly
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("✅ Model loaded!")

⏳ Loading NLLB-200 translation model...


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Model loaded!


In [44]:
LANGUAGE_CODES = {
    "1":  ("English",    "eng_Latn"),
    "2":  ("Arabic",     "arb_Arab"),
    "3":  ("French",     "fra_Latn"),
    "4":  ("Spanish",    "spa_Latn"),
    "5":  ("Portuguese", "por_Latn"),
    "6":  ("German",     "deu_Latn"),
    "7":  ("Italian",    "ita_Latn"),
    "8":  ("Dutch",      "nld_Latn"),
    "9":  ("Turkish",    "tur_Latn"),
    "10": ("Russian",    "rus_Cyrl"),
    "11": ("Chinese",    "zho_Hans"),
    "12": ("Japanese",   "jpn_Jpan"),
    "13": ("Korean",     "kor_Hang"),
    "14": ("Hindi",      "hin_Deva"),
    "15": ("Swedish",    "swe_Latn"),
}


text_input = input("\n📝 Enter text to translate:\n> ").strip()

# Predict source language using Logistic Regression model
preprocessed_text_input = preprocessing(text_input)
X_text_input = vector.transform([preprocessed_text_input])
predicted_src_language_name = lr.predict(X_text_input)[0]

# Find the corresponding NLLB language code for the predicted language
src_code = None
src_key = None
for key, (name, code) in LANGUAGE_CODES.items():
    if name == predicted_src_language_name:
        src_name = name
        src_code = code
        src_key = key
        break

if src_code is None:
    print(f"❌ Error: Predicted source language '{predicted_src_language_name}' not supported by NLLB model. Please ensure the model supports this language.")
else:
    print(f"\n🧠 Detected source language: {src_name}")

    print("\n🎯 Target language (translate TO):\n" +
          "\n".join(f"   {k:>2}. {name}" for k, (name, _) in LANGUAGE_CODES.items()))
    tgt_choice = input("\n> ").strip()

    if tgt_choice not in LANGUAGE_CODES:
        print("❌ Invalid target language choice. Please run the cell again.")
    elif src_key == tgt_choice:
        print("⚠️ Source and target language are the same!")
    else:
        tgt_name, tgt_code = LANGUAGE_CODES[tgt_choice]
        tokenizer.src_lang = src_code
        inputs = tokenizer(text_input, return_tensors="pt")
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_code)

        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_length=512
        )

        translation = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
        print("\n" + "=" * 55)
        print(f"  🔤 Original  ({src_name}): {text_input}")
        print(f"  ✅ Translated ({tgt_name}): {translation}")
        print("=" * 55)


📝 Enter text to translate:
> Ik zou graag over de hele wereld reizen

🧠 Detected source language: Dutch

🎯 Target language (translate TO):
    1. English
    2. Arabic
    3. French
    4. Spanish
    5. Portuguese
    6. German
    7. Italian
    8. Dutch
    9. Turkish
   10. Russian
   11. Chinese
   12. Japanese
   13. Korean
   14. Hindi
   15. Swedish

> 10

  🔤 Original  (Dutch): Ik zou graag over de hele wereld reizen
  ✅ Translated (Russian): Я бы хотел путешествовать по всему миру
